# Detailed `qwanaflow` workflow

## Typical use of qwanaflow

For most users, obtaining a dataset of cell measurements using the `qwanaflow` command should be sufficient for their analysis needs. The options for this command can be displayed as such:

In [ ]:
# Here, the exclamation mark indicates that this command is run by the shell and not by python
# Indeed, qwanaflow should be launched from the command line
!qwanaflow --help  

For example, the complete workflow may be launched from the command line using the following command:

In [ ]:
# Again, the exclamation mark here indicates that this command is passed to the shell
!qwanaflow ../tests/data/test_image.png test_output

Sometimes, however, the user may want more control over the analytical workflow. If this is your case, or if you simply want to better understand what `qwanaflow` does under the hood, please read on. This detailed discussion will also allow you to better understand the parameters passed as arguments to `qwanaflow`.

## Test image

The image we will use as a test image for this example is displayed below. It shows a few growth rings from a subfossil *Picea mariana* sample.

<img src="../tests/data/test_image.png" width=500 height=500>

The image is 3000x3000 pixels, which is smaller than the typical images generated in quantitative wood anatomy, but suitable for demonstration purposes

## Importing the qwanamiz utilities and other required packages

Below, we load the `qwanamiz` that we will need for this example as well as some other packages that are needed for the analysis.

In [ ]:
import qwanamiz.qwanamiz as qmiz                 # main qwanamiz functions
import pandas as pd                              # manipulate data in tabular format with DataFrames
import matplotlib.pyplot as plt                  # plotting library
import skimage.io                                # image input/output from scikit-image
import skimage.measure                           # image measurements from scikit-image
import skimage.segmentation                      # image segmentation from scikit-image
from scipy.ndimage import distance_transform_edt # function that computes distance to nearest lumen pixel
import os                                        # operating-system related utilities
%matplotlib inline

Next, we can read the image as a numpy array under the name `bw_image` (stands for black & white image). Zeros and ones correspond to lumen and cell wall pixels, respectively.

In [ ]:
bw_image_path = os.path.join("..", "tests", "data", "test_image.png")
bw_image = skimage.io.imread(bw_image_path)
bw_image

A critical step of the analysis workflow is labelling the image, which is performed using functionality provided by scikit-image. This step labels all the lumens from individual cells and assigns a distinct integer to each.

In [ ]:
labeled_image = skimage.measure.label(bw_image)
plt.imshow(labeled_image)

Once the cells are labeled, we can compute some lumen properties using scikit-image's `regionprops_table` function. Many properties are available and described in the [official documenation](https://scikit-image.org/docs/0.25.x/api/skimage.measure.html#skimage.measure.regionprops_table), but here we limit ourselves to the ones needed for the purposes of downstream analyses. Prior to computation, we need to set the scaling from pixels to micrometers so we get measurements in micrometers. Here, this scaling parameter is 0.55043 micrometers per pixel because the resolution of our image is 46146 dpi. In typical `qwanaflow` usage, this value would be provided as a command-line argument.

In [ ]:
# Setting the scaling from pixels to micrometers
pix_to_um = 0.55043

# Computing the properties of the cell lumens and returning a pandas DataFrame
regionprops_df = pd.DataFrame(skimage.measure.regionprops_table(
        labeled_image,
        spacing = pix_to_um,
        properties = ('label', 'area', 'major_axis_length', 'minor_axis_length',
                      'centroid', 'orientation', 'perimeter_crofton', 'image',
                      'bbox', 'solidity')))

regionprops_df.info()

Once we have these measurements, we can further refine the set of identified cells. Because the image binzarization step is often imperfect, it is likely that groups of two or more distinct cells will appear merged together and show up in the output as large cells with an irregular shape. We can identify incorrectly merged cells using those properties and use a watershed-based segmentation algorithm to properly redefine cell lumen boundaries.

In [ ]:
# Cells with area > 500 squared micrometers and solidity < 0.95 will be considered by the watershed algorithm
# This function updates the labeled_image and regionprops_df with redefined values
# watershed_result is an integer array that identifies the cells whose ID was potentially redefined
labeled_image, regionprops_df, watershed_result = qmiz.adjust_labels(labeled_image, regionprops_df, scale = pix_to_um,
                                                                     area_threshold = 500, solidity_threshold = 0.95)

# Zooming in on three cells that were split by the segmentation algorithm
plt.imshow(watershed_result[2130:2250,820:1000])

As we can see from the brighter yellow cells in the `labeled_image` array, several cells had their lumens redefined this way by the segmentation algorithm.

In [ ]:
# The cells that were considered by the watershed algorithm are a brighter yellow because their index
# is greater than those of the cells that were defined by the original labeling
plt.imshow(labeled_image)

By now, we already have several of the measurements that one would expect from a quantitative wood anatomy dataset. However, we have yet to measure cell wall thickness, which we will do later in this workflow. However, in preparation for this measurement, we can computed an array that maps each cell wall pixel (i.e. each pixel that is not a lumen pixel) to its distance to the nearest cell lumen. this map will be essential to computing cell wall thickness later on.

In [ ]:
# distance_map: array containing the distance to the nearest lumen pixel
# nearest_label_coords: an array containing the ID (label) of the nearest lumen
distance_map, nearest_label_coords = distance_transform_edt(labeled_image == 0,
                                                            sampling = pix_to_um,
                                                            return_indices = True)

plt.imshow(distance_map)

Based on the distance map and the map of the cell ID nearest to each cell wall pixel, we can expand the cells beyond their lumen to also include their cell walls.

In [ ]:
# The distance parameters indicates that two cell lumens further away than this
# value will not be considered adjacent
expanded_labels = qmiz.expand_cells(labeled_image,
                                    distance_map,
                                    nearest_label_coords,
                                    distance = 10,
                                    spacing = pix_to_um)

plt.imshow(expanded_labels)
